# EPCLM Core-001 — minimal GPU pilot
Use the existing private bingjiangli/minicells Notebook with GPU, Internet, KAGGLE_API_TOKEN and GITHUB_TOKEN. Save & Run All runs the small CPU integration test and only C1: batch 4, TBPTT 64, K4, 50 real Trainer updates (12,800 tokens). At update 25 it saves and resumes into a fresh model/optimizer. It writes pilot-report.json, estimates 250K-token training cost and publishes the pilot report. It does not start development training, other arms, validation, ablations, Inspector or Bootstrap. All scientific hypotheses remain NOT_EVALUATED.


In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-core-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-epclm-core-001'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()
    except Exception:
        raise RuntimeError('Enable the existing Kaggle Secret KAGGLE_API_TOKEN for versioned Output recovery.') from None
if not os.environ['KAGGLE_API_TOKEN']:
    raise RuntimeError('Enable the existing Kaggle Secret KAGGLE_API_TOKEN for versioned Output recovery.')

if not os.environ.get('HF_TOKEN', '').strip():
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception:
        pass  # Optional: the fixed public dataset supports anonymous access.

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix':
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Hard two-hour limit reached. Kaggle Output remains under ' + '/kaggle/working/epclm-core-001-pilot')
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Kaggle Output remains under ' + '/kaggle/working/epclm-core-001-pilot')

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO)])

import importlib.util
for module in ('datasets','tokenizers'):
    if importlib.util.find_spec(module) is None:
        run_bounded([sys.executable, '-m', 'pip', 'install', '--quiet', module], timeout_seconds=180)

run_bounded([sys.executable, str(REPO / "research/projects/cellular-language-models/experiments/epclm-core-001/run.py"), "--pilot-only"], cwd=REPO, timeout_seconds=6000)
